# Simplicits with Franka Robot

This notebook demonstrates a coupled robot–soft-body simulation. A Franka Emika Panda
robot arm (driven by `SolverFeatherstone`) manipulates two deformable Simplicits cubes
(driven by `SimplicitsSolver`) using Jacobian-based inverse kinematics. The robot follows
a pre-scripted sequence of end-effector key poses while the soft bodies deform on contact.

## Imports

We import Kaolin, Newton, Warp, and **k3d** for interactive 3D visualization.
`SolverFeatherstone` drives articulated robot dynamics; `eval_fk` computes
forward kinematics for Jacobian computation. `newton.utils.download_asset`
fetches the Franka URDF.

In [ ]:
from types import SimpleNamespace
import os
import threading

import k3d
import numpy as np
import torch
import warp as wp
from IPython.display import display
from ipywidgets import Button, VBox

import newton
import newton.utils
from newton import Model, ModelBuilder, State, eval_fk
from newton.solvers import SolverFeatherstone
from newton.math import transform_twist

import kaolin
from kaolin.physics.simplicits import SimplicitsObject
from examples.tutorial.tutorial_common import COMMON_DATA_DIR
from kaolin.experimental.newton.builder import SimplicitsModelBuilder
from kaolin.experimental.newton.solver import SimplicitsSolver

wp.clear_kernel_cache()

## Configuration

Key constants:
- **`SOFT_YOUNGS_MODULUS` / `POISSON_RATIO` / `DENSITY`** — elastic material parameters.
- **`FRAME_DT`** — simulation timestep (1/60 s per frame, 1 substep).
- **`COLLISION_PARTICLE_RADIUS`** — radius for Simplicits particle collision detection.
- **`NULL_SPACE_GAIN`** — gain for null-space posture control.
- **`GRIPPER_CLAMP_CLOSE/OPEN`** — gripper activation values in the key pose table.

In [ ]:
# Simulation parameters
FRAME_DT = 1.0 / 60.0
FLOOR_PLANE = 0.0

# Simplicits parameters
SOFT_YOUNGS_MODULUS = 1e5
POISSON_RATIO = 0.45
DENSITY = 500
APPROX_VOLUME = 0.5
NUM_SAMPLES = 10000

# Contact parameters
SOFT_CONTACT_RADIUS = 0.01
SOFT_CONTACT_MARGIN = 0.05
SOFT_CONTACT_KE = 10000
SOFT_CONTACT_KD = 2e-3
ROBOT_FRICTION = 1.0
SELF_CONTACT_FRICTION = 0.25
SOFT_CONTACT_MAX = 1000000

# Collision parameters
COLLISION_PARTICLE_RADIUS = 0.1
DETECTION_RATIO = 1.5
IMPENETRABLE_BARRIER_RATIO = 0.25
COLLISION_PENALTY = 1000.0
MAX_CONTACT_PAIRS = 10000
COLLISION_FRICTION = 0.5

# Robot control parameters
NULL_SPACE_GAIN = 1.0
GRIPPER_CLAMP_CLOSE = 0.06
GRIPPER_CLAMP_OPEN = 0.8

## Warp Kernel and Jacobian

- **`compute_ee_delta`** — GPU kernel that computes the 6-DOF spatial error
  (position + rotation) between the current end-effector pose and the target.
- **`compute_body_jacobian`** — pure-Python function using `wp.Tape` autodiff
  to compute the velocity Jacobian of the end effector w.r.t. joint velocities.

In [ ]:
@wp.kernel
def compute_ee_delta(
    body_q: wp.array(dtype=wp.transform),
    offset: wp.transform,
    body_id: int,
    bodies_per_env: int,
    target: wp.transform,
    ee_delta: wp.array(dtype=wp.spatial_vector),
):
    """Compute the pose difference between end effector and target."""
    env_id = wp.tid()
    tf = body_q[bodies_per_env * env_id + body_id] * offset
    pos = wp.transform_get_translation(tf)
    pos_des = wp.transform_get_translation(target)
    pos_diff = pos_des - pos
    rot = wp.transform_get_rotation(tf)
    rot_des = wp.transform_get_rotation(target)
    ang_diff = rot_des * wp.quat_inverse(rot)
    ee_delta[env_id] = wp.spatial_vector(
        pos_diff[0], pos_diff[1], pos_diff[2], ang_diff[0], ang_diff[1], ang_diff[2]
    )


def compute_body_jacobian(
    model: Model,
    joint_q: wp.array,
    joint_qd: wp.array,
    body_id,
    offset=None,
    velocity: bool = True,
    include_rotation: bool = False,
):
    """Compute the Jacobian of a body with respect to joint coordinates via autodiff."""
    if isinstance(body_id, str):
        body_id = model.body_name.get(body_id)
    if offset is None:
        offset = wp.transform_identity()

    joint_q.requires_grad = True
    joint_qd.requires_grad = True

    if velocity:
        @wp.kernel
        def compute_body_out(body_qd: wp.array(dtype=wp.spatial_vector), body_out: wp.array(dtype=float)):
            mv = transform_twist(offset, body_qd[body_id])
            if wp.static(include_rotation):
                for i in range(6):
                    body_out[i] = mv[i]
            else:
                for i in range(3):
                    body_out[i] = mv[3 + i]
        in_dim = model.joint_dof_count
        out_dim = 6 if include_rotation else 3
    else:
        @wp.kernel
        def compute_body_out(body_q: wp.array(dtype=wp.transform), body_out: wp.array(dtype=float)):
            tf = body_q[body_id] * offset
            if wp.static(include_rotation):
                for i in range(7):
                    body_out[i] = tf[i]
            else:
                for i in range(3):
                    body_out[i] = tf[i]
        in_dim = model.joint_coord_count
        out_dim = 7 if include_rotation else 3

    out_state = model.state(requires_grad=True)
    body_out = wp.empty(out_dim, dtype=float, requires_grad=True)
    tape = wp.Tape()
    with tape:
        eval_fk(model, joint_q, joint_qd, out_state)
        wp.launch(compute_body_out, 1, inputs=[
                  out_state.body_qd if velocity else out_state.body_q], outputs=[body_out])

    def onehot(i):
        x = np.zeros(out_dim, dtype=np.float32)
        x[i] = 1.0
        return wp.array(x)

    J = np.empty((out_dim, in_dim), dtype=wp.float32)
    for i in range(out_dim):
        tape.backward(grads={body_out: onehot(i)})
        J[i] = joint_qd.grad.numpy() if velocity else joint_q.grad.numpy()
        tape.zero()
    return J.astype(np.float32)

## Simplicits Cube Object

`get_simplicits_cube()` loads a unit cube, scales by 0.25, samples interior points,
and creates a single handle `SimplicitsObject`. Returns `(sim_obj, mesh)` so that `mesh`
vertices and faces can be reused for k3d visualization.

In [ ]:
def get_simplicits_cube():
    """Load cube mesh and create a single handle Simplicits object. Returns (sim_obj, mesh)."""
    mesh_path = os.path.join(COMMON_DATA_DIR, "meshes")
    mesh = kaolin.io.import_mesh(
        mesh_path + "/cube.obj", triangulate=True
    ).to('cuda')
    mesh.vertices = kaolin.ops.pointcloud.center_points(
        mesh.vertices.unsqueeze(0), normalize=True
    ).squeeze(0) * 0.25
    orig_vertices = mesh.vertices.clone()

    uniform_pts = torch.rand(NUM_SAMPLES, 3, device='cuda') * (
        orig_vertices.max(dim=0).values - orig_vertices.min(dim=0).values
    ) + orig_vertices.min(dim=0).values

    yms = torch.full((NUM_SAMPLES,), SOFT_YOUNGS_MODULUS, device='cuda')
    prs = torch.full((NUM_SAMPLES,), POISSON_RATIO, device='cuda')
    rhos = torch.full((NUM_SAMPLES,), DENSITY, device='cuda')

    sim_obj = SimplicitsObject.create_rigid(
        uniform_pts, yms, prs, rhos, APPROX_VOLUME
    ) # Single handle objects acts as a affinely deformable at low yms. At higher stiffness it behaves rigidly.
    return sim_obj, mesh


sim_obj, mesh = get_simplicits_cube()
orig_vertices = mesh.vertices.clone()
print(f"Mesh: {mesh.vertices.shape[0]} vertices | Sim samples: {sim_obj.pts.shape[0]}")

## Robot Articulation

`create_articulation(builder)` downloads the Franka Emika Panda URDF via
`newton.utils.download_asset`, adds it to a `ModelBuilder`, sets the initial
joint configuration, and defines the sequence of end-effector key poses used
for the scripted manipulation task.

Returns `(endeffector_id, endeffector_offset, robot_key_poses, targets,
transition_duration, robot_key_poses_time)` for use by the IK controller.

In [ ]:
def create_articulation(builder):
    """Load Franka URDF, set initial pose, define key poses. Returns control metadata."""
    asset_path = newton.utils.download_asset("franka_emika_panda")
    builder.add_urdf(
        str(asset_path / "urdf" / "fr3_franka_hand.urdf"),
        xform=wp.transform(
            (-0.5, -0.5, -0.0),
            wp.quat_identity(),
        ),
        floating=False,
        scale=2,
        enable_self_collisions=False,
        collapse_fixed_joints=True,
        force_show_colliders=False,
    )
    builder.joint_q[:6] = [0.0, 0.0, 0.0, -1.59695, 0.0, 2.5307]

    robot_key_poses = np.array(
        [
            [2.5, 0.31, -0.60, 0.23, 1, 0.0, 0.0, 0.0, GRIPPER_CLAMP_OPEN],
            [2,   0.31, -0.60, 0.23, 1, 0.0, 0.0, 0.0, GRIPPER_CLAMP_CLOSE]
        ],
        dtype=np.float32,
    )
    targets = robot_key_poses[:, 1:]
    transition_duration = robot_key_poses[:, 0]
    robot_key_poses_time = np.cumsum(robot_key_poses[:, 0])
    endeffector_id = builder.body_count - 3
    endeffector_offset = wp.transform([0.0, 0.0, 0.22], wp.quat_identity())
    return endeffector_id, endeffector_offset, robot_key_poses, targets, transition_duration, robot_key_poses_time


franka = ModelBuilder()
endeffector_id, endeffector_offset, robot_key_poses, targets, transition_duration, robot_key_poses_time = \
    create_articulation(franka)

print(f"Franka: {franka.body_count} bodies, {franka.joint_dof_count} DOFs, "
      f"endeffector_id={endeffector_id}")

## Building the Combined Model

`SimplicitsModelBuilder` holds both the soft-body Simplicits objects and the
Franka robot articulation. After adding both, `finalize()` compiles all geometry
and physics parameters into GPU arrays. `state_0` / `state_1` form the double
buffer; `control` holds target joint inputs.

In [ ]:
simplicits_builder = SimplicitsModelBuilder()

simplicits_builder.add_simplicits_object(
    sim_obj,
    init_transform=torch.tensor([
        [1, 0, 0, -0.5],
        [0, 1, 0, -0.5],
        [0, 0, 1,  1.5],
        [0, 0, 0,  1.0]
    ], dtype=torch.float32, device='cuda')
)
simplicits_builder.add_simplicits_object(
    sim_obj,
    init_transform=torch.tensor([
        [1, 0, 0, -0.5],
        [0, 1, 0, -0.5],
        [0, 0, 1,  2.5],
        [0, 0, 0,  1.0]
    ], dtype=torch.float32, device='cuda')
)

simplicits_builder.add_simplicits_collisions(
    collision_particle_radius=COLLISION_PARTICLE_RADIUS,
    detection_ratio=DETECTION_RATIO,
    impenetrable_barrier_ratio=IMPENETRABLE_BARRIER_RATIO,
    collision_penalty=COLLISION_PENALTY,
    max_contact_pairs=MAX_CONTACT_PAIRS,
    friction=COLLISION_FRICTION
)

# Add Franka robot and ground plane
simplicits_builder.add_builder(franka)
bodies_per_env = franka.body_count
dof_q_per_env = franka.joint_coord_count
dof_qd_per_env = franka.joint_dof_count

simplicits_builder.add_ground_plane()

# Finalize model
model = simplicits_builder.finalize(requires_grad=False)
model.soft_contact_ke = SOFT_CONTACT_KE
model.soft_contact_kd = SOFT_CONTACT_KD
model.soft_contact_mu = SELF_CONTACT_FRICTION

state_0 = model.state()
state_1 = model.state()
control = model.control()
target_joint_qd = wp.empty_like(state_0.joint_qd)

print(f"Model: {model.particle_count} particles, {model.body_count} bodies")

## Control Setup and IK

`set_up_control()` allocates the Jacobian computation buffers and defines the
`compute_body_out_kernel` that extracts the end-effector velocity for autodiff.

`generate_control_joint_qd(state_in, sim_time)` computes target joint velocities
via Jacobian pseudo-inverse IK plus a null-space term to maintain a reference posture.
Gripper joints are controlled directly from the current key pose activation value.

In [ ]:
def set_up_control():
    """Allocate IK control buffers and define the end-effector velocity kernel."""
    out_dim = 6
    in_dim = model.joint_dof_count

    def _onehot(i, n):
        return wp.array([1.0 if j == i else 0.0 for j in range(n)], dtype=float)

    Jacobian_one_hots = [_onehot(i, out_dim) for i in range(out_dim)]

    @wp.kernel
    def _compute_body_out(
        body_qd: wp.array(dtype=wp.spatial_vector),
        body_out_arr: wp.array(dtype=float)
    ):
        mv = transform_twist(wp.static(endeffector_offset), body_qd[wp.static(endeffector_id)])
        for i in range(6):
            body_out_arr[i] = mv[i]

    compute_body_out_kernel = _compute_body_out
    temp_state_for_jacobian = model.state(requires_grad=True)
    body_out = wp.empty(out_dim, dtype=float, requires_grad=True)
    J_flat = wp.empty(out_dim * in_dim, dtype=float)
    ee_delta = wp.empty(1, dtype=wp.spatial_vector)
    initial_pose = model.joint_q.numpy()

    return SimpleNamespace(
        one_hots=Jacobian_one_hots,
        kernel=compute_body_out_kernel,
        temp_state=temp_state_for_jacobian,
        body_out=body_out,
        J_flat=J_flat,
        ee_delta=ee_delta,
        initial_pose=initial_pose,
    )


ik = set_up_control()


def generate_control_joint_qd(state_in, sim_time: float):
    """Compute target joint velocities via Jacobian IK + null-space posture control."""
    t_mod = (
        sim_time
        if sim_time < robot_key_poses_time[-1]
        else sim_time % robot_key_poses_time[-1]
    )
    include_rotation = True
    current_interval = np.searchsorted(robot_key_poses_time, t_mod)
    target = targets[current_interval]

    wp.launch(
        compute_ee_delta,
        dim=1,
        inputs=[
            state_in.body_q,
            endeffector_offset,
            endeffector_id,
            bodies_per_env,
            wp.transform(*target[:7]),
        ],
        outputs=[ik.ee_delta],
    )

    # Compute Jacobian via autodiff
    state_in.joint_q.requires_grad = True
    state_in.joint_qd.requires_grad = True
    in_dim = model.joint_dof_count
    tape = wp.Tape()
    with tape:
        eval_fk(model, state_in.joint_q, state_in.joint_qd, ik.temp_state)
        wp.launch(
            ik.kernel, 1,
            inputs=[ik.temp_state.body_qd],
            outputs=[ik.body_out]
        )
    out_dim = 6
    for i in range(out_dim):
        tape.backward(grads={ik.body_out: ik.one_hots[i]})
        wp.copy(ik.J_flat[i * in_dim: (i + 1) * in_dim], state_in.joint_qd.grad)
        tape.zero()

    J = ik.J_flat.numpy().reshape(-1, in_dim)
    delta_target = ik.ee_delta.numpy()[0]
    J_inv = np.linalg.pinv(J)

    I = np.eye(J.shape[1], dtype=np.float32)
    N = I - J_inv @ J

    q = state_in.joint_q.numpy()
    q_des = q.copy()
    q_des[1:] = ik.initial_pose[1:]
    delta_q_null = NULL_SPACE_GAIN * (q_des - q)

    delta_q = J_inv @ delta_target + N @ delta_q_null

    # Gripper control
    delta_q[-2] = target[-1] * 0.04 - q[-2]
    delta_q[-1] = target[-1] * 0.04 - q[-1]

    target_joint_qd.assign(delta_q)

## Solvers

Two solvers share the same model:

- **`robot_solver`** (`SolverFeatherstone`) — articulated dynamics for the Franka arm.
  `model.particle_count` is temporarily set to 0 during its step so it skips soft-body
  particles. Gravity is set to zero during robot stepping (robot uses PD/IK control).
- **`simplicits_solver`** (`SimplicitsSolver`) — Newton optimization for the soft cubes.

Pre-allocated `gravity_zero` / `gravity_earth` arrays avoid CPU allocation in the loop.

In [ ]:
robot_solver = SolverFeatherstone(model, update_mass_matrix_interval=1)
simplicits_solver = SimplicitsSolver(model)

gravity_zero = wp.zeros(1, dtype=wp.vec3)
gravity_earth = wp.array(wp.vec3(0.0, 0.0, -9.81), dtype=wp.vec3)

# Evaluate initial forward kinematics
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

sim_substeps = 1
frame_dt = FRAME_DT
sim_dt = frame_dt / sim_substeps

print("Solvers ready")

## Simulation Loop

Each call to `simulate()` first computes target joint velocities via IK, then
runs `sim_substeps` physics steps:

1. **Robot substep** — `model.particle_count=0` and `gravity=0` so the Featherstone
   solver only integrates the articulated chain. `shape_contact_pair_count=0` skips
   rigid–rigid contacts during this pass.
2. **Simplicits substep** — collide + Newton solve for the two soft cubes against all
   shapes (including robot bodies).

States are swapped after each substep.

In [ ]:
sim_time = 0.0


def simulate_step():
    global state_0, state_1

    state_0.clear_forces()
    state_1.clear_forces()

    # Robot step: disable particles and gravity
    particle_count = model.particle_count
    model.particle_count = 0
    model.gravity.assign(gravity_zero)
    model.shape_contact_pair_count = 0

    state_0.joint_qd.assign(target_joint_qd)
    robot_solver.step(state_0, state_1, control, None, sim_dt)

    # Restore and run Simplicits
    model.particle_count = particle_count
    model.gravity.assign(gravity_earth)

    contacts = model.collide(state_0)
    simplicits_solver.step(state_0, state_1, None, contacts, sim_dt)

    state_0, state_1 = state_1, state_0


def simulate():
    global sim_time
    generate_control_joint_qd(state_0, sim_time)
    for _ in range(sim_substeps):
        simulate_step()
    sim_time += frame_dt

## Visualization with k3d

Two sets of k3d objects are rendered:

- **`mesh_soft`** (blue) — the two Simplicits cubes combined into a single mesh.
  Updated each frame via `get_object_deformed_pts`.
- **`mesh_robot`** (orange) — one combined mesh for the entire robot, built from
  `model.shape_source` after `finalize()`. `get_robot_mesh_verts` transforms each
  solid MESH shape from body frame to world frame using `body_q` each frame.
  Faces are static and cached once; only vertices are updated per frame.

Play/Stop/Reset buttons control the background simulation thread.

In [ ]:
orig_vertices_np = orig_vertices.cpu().numpy().astype(np.float32)
orig_faces_np = mesh.faces.cpu().numpy().astype(np.uint32)
num_verts = orig_vertices_np.shape[0]

combined_faces_np = np.concatenate(
    [orig_faces_np, orig_faces_np + num_verts]
).astype(np.uint32)

# --- Robot mesh helpers ---


def apply_transform(verts, tf_row):
    """Apply a (tx, ty, tz, qx, qy, qz, qw) transform to an (N, 3) vertex array."""
    trans = tf_row[:3]
    q = torch.from_numpy(tf_row[3:7].astype(np.float32)).unsqueeze(0)
    rot = kaolin.math.quat.rot33_from_quat(q)[0].numpy()
    return verts @ rot.T + trans


def get_robot_mesh_verts(body_q_np):
    """Collect all solid MESH shapes from model, transform to world, return combined verts+faces."""
    shape_body      = model.shape_body.numpy()
    shape_geo_src   = model.shape_source          # Python list, not a Warp array
    shape_geo_type  = model.shape_type.numpy()
    shape_geo_scale = model.shape_scale.numpy()
    shape_transform = model.shape_transform.numpy()
    shape_is_solid  = model.shape_is_solid.numpy()

    all_verts, all_faces, verts_offset = [], [], 0

    for i in range(len(shape_geo_src)):
        if shape_geo_type[i] != newton.GeoType.MESH or not shape_is_solid[i]:
            continue
        v = shape_geo_src[i].vertices.copy().astype(np.float32)
        f = shape_geo_src[i].indices.reshape(-1, 3).astype(np.uint32)

        q = torch.from_numpy(shape_transform[i][3:].astype(np.float32)).unsqueeze(0)
        R = kaolin.math.quat.rot33_from_quat(q)[0].numpy()
        p = shape_transform[i][:3]
        v = v * shape_geo_scale[i].reshape(1, -1)   # per-axis scale
        v = (R @ v.T).T + p                          # shape-local → body frame

        if shape_body[i] >= 0:
            v = apply_transform(v, body_q_np[shape_body[i]])  # body → world

        all_verts.append(v)
        all_faces.append(f + verts_offset)
        verts_offset += len(v)

    return np.concatenate(all_verts, axis=0), np.concatenate(all_faces, axis=0)


# --- Build k3d plot ---

plot = k3d.plot(camera_auto_fit=False)
plot.camera = [3, 3, 3,  0, 0, 0,  0, 0, 1]

mesh_soft = k3d.mesh(
    np.concatenate([orig_vertices_np, orig_vertices_np]).astype(np.float32),
    combined_faces_np,
    color=0x3399ff
)
plot += mesh_soft

# Robot: compute initial verts+faces, cache faces (static)
init_robot_verts, robot_mesh_faces = get_robot_mesh_verts(state_0.body_q.numpy())
mesh_robot = k3d.mesh(init_robot_verts, robot_mesh_faces, color=0xff6633)
plot += mesh_robot

# Floor plane at FLOOR_PLANE height
_s = 3.0  # half-extent
floor_verts = np.array([
    [-_s, -_s, FLOOR_PLANE],
    [ _s, -_s, FLOOR_PLANE],
    [ _s,  _s, FLOOR_PLANE],
    [-_s,  _s, FLOOR_PLANE],
], dtype=np.float32)
floor_faces = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.uint32)
mesh_floor = k3d.mesh(floor_verts, floor_faces, color=0xcccccc, opacity=0.5)
plot += mesh_floor

plot.display()


def update_vis():
    # Soft body cubes
    model.simplicits_scene.sim_z = state_0.sim_z
    v0 = model.simplicits_scene.get_object_deformed_pts(0, orig_vertices)
    v1 = model.simplicits_scene.get_object_deformed_pts(1, orig_vertices)
    mesh_soft.vertices = np.concatenate(
        [v0.cpu().numpy(), v1.cpu().numpy()]
    ).astype(np.float32)
    # Robot mesh (faces are static, only update vertices)
    robot_verts, _ = get_robot_mesh_verts(state_0.body_q.numpy())
    mesh_robot.vertices = robot_verts


sim_running = [False]
sim_thread = [None]


def run_sim_loop():
    while sim_running[0]:
        simulate()
        update_vis()


def on_play(b):
    if not sim_running[0]:
        sim_running[0] = True
        sim_thread[0] = threading.Thread(target=run_sim_loop, daemon=True)
        sim_thread[0].start()


def on_stop(b):
    sim_running[0] = False


def on_reset(b):
    global state_0, state_1, sim_time
    sim_running[0] = False
    if sim_thread[0] is not None:
        sim_thread[0].join()
    model.simplicits_scene.reset_scene()
    state_0 = model.state()
    state_1 = model.state()
    sim_time = 0.0
    update_vis()


buttons = [Button(description=x) for x in ["Play", "Stop", "Reset"]]
buttons[0].on_click(on_play)
buttons[1].on_click(on_stop)
buttons[2].on_click(on_reset)

update_vis()
display(VBox(buttons))